# TP 3 — Nettoyer et tester : un module de transformations PySpark
**Big Data Engineering — Master 1 — DMI/FST/UCAD/ISI — Prof. Samba Ndiaye**

Objectif : transformer `customers.csv` (sale) en une table clients **propre**,
avec un code **modulaire et testé**.

**Consignes**
- Complétez chaque cellule marquée `# === À COMPLÉTER ===` (remplacez les `...`).
- Exécutez le notebook **de bout en bout** sans erreur.
- Poussez le notebook **avec ses sorties** sur votre dépôt GitHub.

Rappel : les fonctions de nettoyage « réelles » vivent dans `src/transformations.py`.
Ce notebook **démontre** et **mesure** ; il importe le module.


## 0. Vérification de l'environnement


In [1]:
import os
import sys

# Forcer l'utilisation de l'exécutable Python actuel pour Spark
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession, functions as F

# Configuration robuste pour éviter "Python worker failed to connect back" sur Windows
spark = (SparkSession.builder
         .master("local[1]")
         .appName("TP3-nettoyage")
         .config("spark.driver.bindAddress", "127.0.0.1")
         .config("spark.driver.host", "127.0.0.1")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
spark


### Tableau de relevés
On consigne ici les mesures au fil du TP (à reporter dans `docs/QUALITE.md`).


In [2]:
releves = {
    "lignes_brutes": None,
    "emails_manquants": None,
    "villes_distinctes_avant": None,
    "villes_distinctes_apres": None,
    "espaces_dans_noms": None,
    "dates_invalides": None,
    "doublons_exacts": None,
    "telephones_invalides": None,
    "lignes_apres_nettoyage": None,
}
releves


{'lignes_brutes': None,
 'emails_manquants': None,
 'villes_distinctes_avant': None,
 'villes_distinctes_apres': None,
 'espaces_dans_noms': None,
 'dates_invalides': None,
 'doublons_exacts': None,
 'telephones_invalides': None,
 'lignes_apres_nettoyage': None}

## 1. Charger avec un schéma explicite
On impose le schéma plutôt que de le laisser deviner (fiabilité + vitesse).


In [3]:
from pyspark.sql.types import StructType, StructField, StringType

schema_clients = StructType([
    StructField("customer_id",     StringType(), False),
    StructField("prenom",          StringType(), True),
    StructField("nom",             StringType(), True),
    StructField("email",           StringType(), True),
    StructField("telephone",       StringType(), True),
    StructField("ville",           StringType(), True),
    StructField("region",          StringType(), True),
    StructField("date_naissance",  StringType(), True),
    StructField("date_inscription",StringType(), True),
])

df_brut = (spark.read.option("header", True)
                 .schema(schema_clients)
                 .csv("../data/customers.csv"))

releves["lignes_brutes"] = df_brut.count()
df_brut.printSchema()
print("lignes :", releves["lignes_brutes"])


root
 |-- customer_id: string (nullable = true)
 |-- prenom: string (nullable = true)
 |-- nom: string (nullable = true)
 |-- email: string (nullable = true)
 |-- telephone: string (nullable = true)
 |-- ville: string (nullable = true)
 |-- region: string (nullable = true)
 |-- date_naissance: string (nullable = true)
 |-- date_inscription: string (nullable = true)

lignes : 5025


## 2. Diagnostic : mesurer les défauts
On **mesure** chaque défaut avant de corriger quoi que ce soit.

### 2.1 Valeurs manquantes par colonne


In [4]:
# CORRIGÉ
df_brut.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_brut.columns
]).show()


+-----------+------+---+-----+---------+-----+------+--------------+----------------+
|customer_id|prenom|nom|email|telephone|ville|region|date_naissance|date_inscription|
+-----------+------+---+-----+---------+-----+------+--------------+----------------+
|          0|     0|  0|   75|        0|    0|     0|             0|               0|
+-----------+------+---+-----+---------+-----+------+--------------+----------------+



### 2.2 Faux manquants (emails "" ou "N/A")


In [5]:
# CORRIGÉ
nb_email_vide = df_brut.filter(
    (F.trim(F.col("email")) == "") | (F.col("email") == "N/A")
    | F.col("email").isNull()
).count()
print("emails vides, N/A ou null :", nb_email_vide)
releves["emails_manquants"] = nb_email_vide
# Observation attendue : de l'ordre de ~3 % des clients.


emails vides, N/A ou null : 150


### 2.3 Villes distinctes (avant normalisation) et doublons exacts


In [6]:
# CORRIGÉ
releves["villes_distinctes_avant"] = df_brut.select("ville").distinct().count()
releves["doublons_exacts"] = df_brut.count() - df_brut.distinct().count()
print(releves["villes_distinctes_avant"], "villes distinctes (brut)")
print(releves["doublons_exacts"], "doublons exacts")
# Observation attendue : bien plus de 19 villes a cause de la casse/accents.


499 villes distinctes (brut)
15 doublons exacts


### 2.4 Autres défauts (espaces, dates, téléphones)


In [7]:
# Espaces dans les noms
releves["espaces_dans_noms"] = df_brut.filter(
    (F.col("prenom") != F.trim(F.col("prenom"))) | 
    (F.col("nom") != F.trim(F.col("nom"))) |
    (F.col("prenom").rlike(r"\s{2,}")) |
    (F.col("nom").rlike(r"\s{2,}"))
).count()

# Dates de naissance invalides (format ou plage 1920-today)
releves["dates_invalides"] = df_brut.filter(
    (F.col("date_naissance").isNull()) |
    (F.to_date(F.col("date_naissance"), "yyyy-MM-dd").isNull()) |
    (F.col("date_naissance") < "1920-01-01") |
    (F.col("date_naissance") > F.date_format(F.current_date(), "yyyy-MM-dd"))
).count()

# Téléphones invalides (Sénégal : 70/75/76/77/78 + 7 chiffres)
tel_clean = F.regexp_replace(F.col("telephone"), r"[\s\-\.]", "")
tel_clean = F.regexp_replace(tel_clean, r"^\+221", "")
tel_clean = F.regexp_replace(tel_clean, r"^00221", "")
releves["telephones_invalides"] = df_brut.withColumn("tel_clean", tel_clean).filter(
    (F.col("telephone").isNull()) |
    (~F.col("tel_clean").rlike(r"^(70|75|76|77|78)\d{7}$"))
).count()

print(f"Espaces noms : {releves['espaces_dans_noms']}")
print(f"Dates invalides : {releves['dates_invalides']}")
print(f"Téléphones invalides : {releves['telephones_invalides']}")


Espaces noms : 100
Dates invalides : 5025
Téléphones invalides : 0


## 3. Les fonctions de transformation (dans src/)
Rappel : les fonctions de nettoyage « réelles » vivent dans `src/transformations.py`.
On les importe ici pour garantir la cohérence entre le notebook et les tests.


In [8]:
import sys
import os
# Ajouter le dossier src au path pour pouvoir importer transformations
sys.path.append(os.path.abspath("../src"))

from transformations import (
    unifier_manquants,
    normaliser_email,
    normaliser_ville,
    normaliser_telephone,
    valider_naissance,
    dedupliquer_clients,
    nettoyer_clients
)


## 4. Assembler le pipeline et mesurer l'effet


In [9]:
# Section 4 : Mesurer l'effet du nettoyage
df_net = nettoyer_clients(df_brut)

releves["villes_distinctes_apres"] = df_net.select("ville_norm").distinct().count()
releves["lignes_apres_nettoyage"]  = df_net.count()
print("avant :", releves["lignes_brutes"], "-> apres :", releves["lignes_apres_nettoyage"])
print("villes distinctes :", releves["villes_distinctes_avant"],
      "->", releves["villes_distinctes_apres"])


avant : 5025 -> apres : 5000
villes distinctes : 499 -> 499


### 4.1 Vérification visuelle : top des villes après nettoyage


In [10]:
df_net.groupBy("ville_norm").count().orderBy(F.desc("count")).show(10)


+--------------------+-----+
|          ville_norm|count|
+--------------------+-----+
|           rue gomes|   20|
|    97, avenue robin|   19|
|71, avenue mathil...|   19|
|     55, rue laurent|   18|
|936, boulevard de...|   18|
|      561, rue perez|   18|
| 53, boulevard louis|   17|
|  avenue david faure|   17|
|  1, chemin valentin|   17|
|309, avenue de le...|   17|
+--------------------+-----+
only showing top 10 rows



### 4.2 Tableau de relevés final


In [11]:
for k, v in releves.items():
    print(f"{k:30s} : {v}")


lignes_brutes                  : 5025
emails_manquants               : 150
villes_distinctes_avant        : 499
villes_distinctes_apres        : 499
espaces_dans_noms              : 100
dates_invalides                : 5025
doublons_exacts                : 15
telephones_invalides           : 0
lignes_apres_nettoyage         : 5000


## 5. Questions de réflexion
Répondez en quelques lignes (cellule markdown ci-dessous).

1. Combien de villes distinctes **avant** et **après** normalisation ? Que
   conclure sur l'effet de la casse et des accents ?
2. Quelle décision avez-vous prise pour les emails manquants (drop ou fill) ?
   Pourquoi ?
3. Vous avez écrit une **UDF** (`sans_accent`). À quel coût ? Pourquoi est-elle
   justifiée ici alors que la règle est « fonctions intégrées d'abord » ?
4. En quoi la déduplication **après** normalisation diffère-t-elle d'une
   déduplication naïve ?


## 6. Vers le livrable
1. Déplacez les fonctions de la section 3 dans `src/transformations.py`.
2. Écrivez les tests dans `tests/test_transformations.py` (+ `conftest.py`).
3. Vérifiez `pytest -q` : **tout au vert**.
4. Remplissez `docs/QUALITE.md` avec le tableau de relevés.
5. Poussez le tout :

```bash
git add src/ tests/ notebooks/ docs/
git commit -m "feat: module de nettoyage clients + tests (TP3)"
git push
```

> Rappel : `data/` n'est **jamais** commité.


In [12]:
# Arret propre de la session Spark
spark.stop()
print("Session fermee. Notebook termine.")


Session fermee. Notebook termine.
